In [ ]:
!pip install torch transformers datasets accelerate bitsandbytes peft unsloth

In [9]:
%%writefile main.py
import torch
from unsloth import FastLanguageModel
from transformers import TrainingArguments, Trainer
from peft import LoraConfig
from transformers import DataCollatorForLanguageModeling
import os

os.environ["WANDB_DISABLED"] = "true"

# 1. Load LLaMA-3.2 1B in 4-bit mode
model_name = "meta-llama/Llama-3.2-1b"
max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,
    load_in_4bit=True,
    max_seq_length=max_seq_length
)

# 2. Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_rslora=False,
)

# prepare some fake data for testing
fake_data = [
    {"text": "Artificial intelligence is evolving rapidly."},
    {"text": "Machine learning enables computers to learn patterns."},
    {"text": "Natural language processing improves human-computer interaction."}
]


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_seq_length)

tokenized_fake_data = list(map(tokenize_function, fake_data))

# 4. Training Setup
training_args = TrainingArguments(
    output_dir="./llama-unsloth",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    learning_rate=2e-4,
    num_train_epochs=1,
    save_strategy="epoch",
    optim="adamw_torch",
    fp16=True,
    logging_steps=10,
    disable_tqdm=True
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_fake_data,
    eval_dataset=tokenized_fake_data,
    data_collator=data_collator,
)


Overwriting main.py


In [10]:
%%writefile test_main.py
import pytest
import torch
from main import trainer, model, tokenizer

def test_trainer_initialization():
    """Ensure the Trainer is properly initialized."""
    assert trainer is not None, "Trainer object is not initialized"
    assert trainer.model is not None, "Trainer model is missing"

def test_model_device():
    """Check if the model is on the correct device (CPU or GPU)."""
    device = next(model.parameters()).device
    assert device.type in ["cuda", "cpu"], f"Unexpected device: {device}"

def test_training_step():
    """Run a simple training step to ensure trainer execution."""
    try:
        trainer.train()
    except Exception as e:
        pytest.fail(f"Trainer failed to execute: {e}")

def test_tokenizer():
    """Ensure the tokenizer processes text correctly."""
    sample_text = "Hello, world!"
    tokens = tokenizer(sample_text, return_tensors="pt")
    assert "input_ids" in tokens, "Tokenizer did not return input IDs"

if __name__ == "__main__":
    pytest.main()


Overwriting test_main.py


In [11]:
!pytest test_main.py

======================================= test session starts ========================================
platform linux -- Python 3.11.11, pytest-8.3.4, pluggy-1.5.0
rootdir: /content
plugins: anyio-3.7.1, langsmith-0.3.11, typeguard-4.4.2
collected 4 items                                                                                  

test_main.py ....                                                                            [100%]

======================================== 4 passed in 41.58s ========================================
